# Ingest the current five-minute files

This notebook reads the changing DispatchIS current listing, downloads files not already present and extracts their CSVs into the current Bronze path.

### Storage prerequisite

Before running the pipeline, I created the Databricks Volume `/Volumes/workspace/default/aemo_mlops_volume`. The ingestion notebooks write their Bronze files there, and the Silver transformation reads from the same location.

## 1. Discover the files currently available

Current filenames include a twelve-digit dispatch timestamp and a run identifier. Parsing the live listing avoids hard-coding a URL that changes every five minutes.

### How the current filename pattern works

```python
r'PUBLIC_DISPATCHIS_\d{12}_[^"\s<>]*\.zip'
```

- `r'...'` makes this a raw Python string, so backslashes are passed directly to the pattern matcher.
- `PUBLIC_DISPATCHIS_` requires the fixed prefix used by current DispatchIS files.
- `\d{12}` requires twelve digits: `YYYYMMDDHHMM`, the date and time of the five-minute interval.
- `_` requires the separator before AEMO's run identifier.
- `[^"\s<>]*` accepts the remaining identifier but stops before quotes, whitespace or HTML tag boundaries.
- `\.zip` requires the literal `.zip` extension.

For example, it accepts a name such as `PUBLIC_DISPATCHIS_202608092005_0000000531643185.zip`. This pattern is more specific than the daily pattern because each current file represents one dispatch interval and includes a run identifier.

In [0]:
import os
import re
import time
import zipfile
from urllib.parse import urljoin

import requests


def get_current_urls():
    folder = (
        "https://www.nemweb.com.au/"
        "REPORTS/CURRENT/DispatchIS_Reports/"
    )

    # Read the current HTML listing because its file set changes throughout the day.
    html = requests.get(folder).text

    # Match timestamped DispatchIS ZIPs and retain the run identifier in the filename.
    files = re.findall(
        r'PUBLIC_DISPATCHIS_\d{12}_[^"\s<>]*\.zip',
        html
    )

    urls = [
        urljoin(folder, file)
        for file in sorted(set(files))
    ]

    return urls

In [0]:
urls = get_current_urls()

print(f"Found {len(urls)} current files")

for url in urls:
    print(url)

## 2. Download only new current ZIPs

Each run compares the listing with the current Bronze folder. Files already present are skipped.

In [0]:
def download_if_not_exists(url, bronze_folder):
    filename = url.split("/")[-1]
    path = os.path.join(bronze_folder, filename)

    if os.path.exists(path):
        print(f"Skipping: {filename}")
        return path

    print(f"Downloading: {filename}")
    start = time.time()

    response = requests.get(url)
    response.raise_for_status()

    # Preserve the source ZIP unchanged in Bronze.
    with open(path, "wb") as file:
        file.write(response.content)

    print(f"Finished: {filename} - {time.time() - start:.1f}s")

    return path

In [0]:
bronze_folder = "/Volumes/workspace/default/aemo_mlops_volume/bronze/current"

os.makedirs(bronze_folder, exist_ok=True)

for url in urls:
    download_if_not_exists(url, bronze_folder)

## 3. Extract the current CSV files

Unlike the daily source, each current ZIP can be extracted directly into the CSV folder.

In [0]:
csv_folder = bronze_folder + "_uncompressed"

os.makedirs(csv_folder, exist_ok=True)

zip_files = [
    file for file in os.listdir(bronze_folder)
    if file.endswith(".zip")
]

start = time.time()

print(f"Found {len(zip_files)} ZIP files\n")

for i, filename in enumerate(sorted(zip_files), 1):

    path = os.path.join(bronze_folder, filename)

    with zipfile.ZipFile(path, "r") as zip_file:

        files = zip_file.namelist()

        already_extracted = all(
            os.path.exists(os.path.join(csv_folder, file))
            for file in files
        )

        if already_extracted:
            print(f"[{i}/{len(zip_files)}] Skipping: {filename}")
            continue

        print(f"[{i}/{len(zip_files)}] Extracting: {filename}")

        zip_file.extractall(csv_folder)

print()
print(f"Finished in {time.time() - start:.1f} seconds")